# Single Fidelity Surrogate Model Comparison

This workbook is aimed at testing the convergence of single fidelity models to data sets taken from analytical functions. The goal is to show how the functions increase in accuracy as the number of samples increase.

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt

import multi_fidelity_functions as mfs
from SurrogateModels import kriging

The following creates some scaffolding for systematically creating and assessing kriging models.

In [2]:
import time
import csv


def create_test_points(func: mfs.MultiFidelityTemplate, n=2000):
    # Create a set of samples to create a ground truth
    X_test = func.sample_random(n)
    y_test = func.high_fidelity(X_test)
    return X_test, y_test


def generate_kriging_model(
    name,
    func: mfs.MultiFidelityTemplate,
    n,
    seed,
    X_test,
    y_test,
    x_transform=True,
    bounds=None,
):
    X_train = func.sample_random(n, seed=seed)
    # X_train = func.sample_uniform(n)
    y_train = func.high_fidelity(X_train)
    # Setup timer to gauge model training time
    t0 = time.perf_counter()
    model = kriging.Kriging(
        X_train,
        y_train,
        nugget=1e-7,
        scale_input=x_transform,
        bounds=bounds,
    )
    model_training_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    error = model.model_error(X_test, y_test)
    evaluation_time = time.perf_counter() - t0
    model_error = model.model_error(X_test, y_test)
    print(
        f"Run {name} :: seed {seed} :: points {n} :: error {error} :: train_time {model_training_time} complete"
    )
    return model, error, model_training_time, evaluation_time


def generate_model_comparisons(
    name,
    func: mfs.MultiFidelityTemplate,
    X_test,
    y_test,
    sample_low=5,
    sample_high=105,
    step_size=5,
    seeds=5,
    points=[],
    model_error=[],
    training_time=[],
    evaluation_time=[],
    iterations = [],
):
    # Generate a Kriging model of the data and test accuracy using a number of random sample points
    n_points = list(range(sample_low, sample_high, step_size))
    for seed in range(0, seeds):
        for n in n_points:
            X_train = func.sample_random(n, seed=seed)
            # X_train = func.sample_uniform(n)
            y_train = func.high_fidelity(X_train)
            # Setup timer to gauge model training time
            t0 = time.perf_counter()
            model = kriging.Kriging(X_train, y_train, nugget=1e-6)
            training_time.append(time.perf_counter() - t0)

            t0 = time.perf_counter()
            error = model.model_error(X_test, y_test)
            evaluation_time.append(time.perf_counter() - t0)
            model_error.append(model.model_error(X_test, y_test))
            points.append(n)
            iterations.append(model.iter)
            print(
                f"Run {name} :: seed {seed} :: points {n} :: error {error} :: train_time {training_time[-1]} :: iterations {model.iter}"
            )
    return points, model_error, evaluation_time, training_time, iterations


def plot_error_measure(
    points, error, name, color="blue", marker="o", label="Test", ax=None
):
    if ax == None:
        fig = plt.figure(name)
        ax = fig.add_subplot()
    ax.set_title(name)
    ax.scatter(
        points,
        error,
        color=color,
        label=label,
        marker=marker,
    )
    ax.set_xlabel("Samples")
    ax.set_ylabel("RMS Error")
    ax.legend()
    ax.grid()
    ax.set_yscale("log")
    return ax


def export_to_csv(output_file, points, model_error, training_time, evaluation_time):
    with open(output_file, "w", newline="") as file:
        writer = csv.writer(file)
        for i, val in enumerate(points):
            writer.writerow(
                [
                    val,
                    model_error[i],
                    training_time[i],
                    evaluation_time[i],
                ]
            )

In [ ]:
key = "2D"
my_function = mfs.increasing_dimensions[key]()
X_test, y_test = create_test_points(my_function, n=3000)

points=[]
model_error=[]
training_time=[]
evaluation_time=[]
iterations = []
model, error, model_training_time, evaluation_times = generate_kriging_model(
    key,
    my_function,
    n=1000,
    X_test=X_test,
    y_test=y_test,
    x_transform=True,
    bounds=my_function.bounds,
    seeds=0,
)

In [5]:
key = "2D"
my_function = mfs.increasing_dimensions[key]()
X_test, y_test = create_test_points(my_function, n=3000)

points=[]
model_error=[]
training_time=[]
evaluation_time=[]
iterations = []
points, model_error, evaluation_time, training_time, iterations = generate_model_comparisons(
    key,
    my_function,
    X_test=X_test,
    y_test=y_test,
    sample_low=150,
    sample_high=1000,
    step_size=100,
    seeds=3,    
    points=points,
    model_error=model_error,
    training_time=training_time,
    evaluation_time=evaluation_time,
    iterations=iterations
)

[{'iter': 0, 'x': array([-3.46965436e-05,  3.46965436e-05,  1.99998843e+00,  1.99998843e+00]), 'f': -239.4958707541063}, {'iter': 1, 'x': array([-3.38735242e-05,  3.38846130e-05,  2.00000000e+00,  1.99998079e+00]), 'f': -239.563366135857}, {'iter': 2, 'x': array([-4.06173018e-05,  4.06318400e-05,  2.00000000e+00,  1.99997793e+00]), 'f': -239.57462801769924}, {'iter': 3, 'x': array([-5.92138332e-05,  5.92371104e-05,  2.00000000e+00,  1.99997094e+00]), 'f': -239.58855423933522}, {'iter': 4, 'x': array([-6.70595352e-05,  6.70864582e-05,  2.00000000e+00,  1.99996901e+00]), 'f': -239.59034165291}, {'iter': 5, 'x': array([-8.03192797e-05,  8.03521591e-05,  2.00000000e+00,  1.99996702e+00]), 'f': -239.59283919306543}, {'iter': 6, 'x': array([-1.15124739e-04,  1.15173220e-04,  2.00000000e+00,  1.99996373e+00]), 'f': -239.59915056056582}, {'iter': 7, 'x': array([-2.06262064e-04,  2.06351700e-04,  2.00000000e+00,  1.99995814e+00]), 'f': -239.61518047351592}, {'iter': 8, 'x': array([-4.42217637e-

KeyboardInterrupt: 

### 4-Dimensional Dataset

In [4]:
key = "2D"
my_function = mfs.increasing_dimensions[key]()
X_test, y_test = create_test_points(my_function, n=3000)
points, model_error, evaluation_time, training_time, iterations = generate_model_comparisons(
    key,
    my_function,
    X_test=X_test,
    y_test=y_test,
    sample_low=1000,
    sample_high=2000,
    step_size=100,
    seeds=1,
)

Run 2D :: seed 0 :: points 1000 :: error 4.6161255827643704e-05 :: train_time 172.1625060000224 :: iterations 9


KeyboardInterrupt: 

In [ ]:
report_name = "SingleFidelity4D_Cokriging_260823.csv"
print(model_error)
export_to_csv(
    report_name,
    points=points,
    model_error=model_error,
    training_time=training_time,
    evaluation_time=evaluation_time,
)

[0.02104515627990947]
